# Notebook 13 — Multi-LoRA Serving and Runtime Model Selection

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/systems/13_multi_lora_serving_and_runtime_model_selection.ipynb)

Serving adapters is the operational sequel to finetuning. Once you have a base model plus several LoRAs, the systems question becomes: which adapters stay hot, which runtime should host them, and when should traffic route to a different model family entirely?

## What you will build

- an open-source registry of base models and LoRA adapters
- traffic classes and tenant mixes for a multi-tenant serving stack
- routing heuristics for base models, adapters, and runtimes
- simulations for base-only, hot-swap, packed multi-LoRA, and tenant-partitioned serving
- a small serving manifest covering policy tradeoffs and runtime recommendations

## Why this notebook follows `finetuning/`

Finetuning creates specialized behavior. Serving decides whether that behavior is actually available under latency, memory, and traffic constraints. A LoRA that looks excellent offline can still be operationally awkward if every request has to cold-load it.

In [ ]:
# --- Systems Notebook Setup ---
!pip install -q "transformers>=4.51.0" accelerate torch numpy pandas matplotlib fastapi uvicorn pydantic httpx psutil

import asyncio
import json
import math
import random
import statistics
import threading
import time
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-systems")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def now_ms():
    return time.perf_counter() * 1000

def summarize(values):
    return {
        "count": len(values),
        "mean": statistics.mean(values) if values else 0.0,
        "median": statistics.median(values) if values else 0.0,
        "min": min(values) if values else 0.0,
        "max": max(values) if values else 0.0,
    }

print("✓ Systems notebook environment ready")
print("  Cache dir:", CACHE_DIR)
print("  Artifact root:", ARTIFACT_ROOT)

## Step 1: Add helpers and an artifact directory

We will keep everything notebook-native: small registries, simple routing heuristics, and synthetic traffic. The goal is to reason about adapter residency and runtime choice without needing a production cluster.

In [ ]:
import copy

random.seed(13)
ARTIFACT_DIR = ARTIFACT_ROOT / "13_multi_lora_serving_and_model_selection"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def percentile(values, pct):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return 0.0
    rank = (len(ordered) - 1) * (pct / 100.0)
    lower = math.floor(rank)
    upper = math.ceil(rank)
    if lower == upper:
        return ordered[int(rank)]
    weight = rank - lower
    return ordered[lower] * (1.0 - weight) + ordered[upper] * weight

def clipped(value, low=0.0, high=1.0):
    return max(low, min(high, value))

def weighted_pick(weight_map, rng):
    roll = rng.random()
    cursor = 0.0
    items = list(weight_map.items())
    for name, weight in items:
        cursor += weight
        if roll <= cursor:
            return name
    return items[-1][0]

def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 2: Define a base-model and adapter registry

We will use open-source references only: `vLLM`, `SGLang`, and `llama.cpp` for the runtimes, plus a small set of realistic adapter targets.

In [ ]:
base_models = pd.DataFrame([
    {"route": "qwen-general-vllm", "model": "Qwen/Qwen2.5-7B-Instruct", "runtime": "vLLM", "base_vram_gib": 15.0, "base_latency_ms": 640, "throughput_tps": 150, "quality_general": 0.86, "quality_code": 0.78, "quality_structured": 0.84, "context_tokens": 32768, "local_capable": False, "supports_multi_lora": True},
    {"route": "llama-structured-sglang", "model": "meta-llama/Meta-Llama-3.1-8B-Instruct", "runtime": "SGLang", "base_vram_gib": 16.5, "base_latency_ms": 700, "throughput_tps": 138, "quality_general": 0.87, "quality_code": 0.79, "quality_structured": 0.90, "context_tokens": 65536, "local_capable": False, "supports_multi_lora": True},
    {"route": "qwen-coder-vllm", "model": "Qwen/Qwen2.5-Coder-7B-Instruct", "runtime": "vLLM", "base_vram_gib": 15.3, "base_latency_ms": 680, "throughput_tps": 142, "quality_general": 0.81, "quality_code": 0.91, "quality_structured": 0.82, "context_tokens": 32768, "local_capable": False, "supports_multi_lora": True},
    {"route": "mistral-edge-llamacpp", "model": "Mistral-7B-Instruct-v0.3-GGUF", "runtime": "llama.cpp", "base_vram_gib": 6.2, "base_latency_ms": 980, "throughput_tps": 42, "quality_general": 0.78, "quality_code": 0.70, "quality_structured": 0.76, "context_tokens": 8192, "local_capable": True, "supports_multi_lora": False},
])

adapters = pd.DataFrame([
    {"adapter": "support-lora", "route": "qwen-general-vllm", "domain": "support", "delta_vram_gib": 0.65, "quality_lift": 0.05, "cold_load_ms": 240},
    {"adapter": "finance-lora", "route": "qwen-general-vllm", "domain": "finance", "delta_vram_gib": 0.74, "quality_lift": 0.06, "cold_load_ms": 300},
    {"adapter": "sql-lora", "route": "qwen-coder-vllm", "domain": "sql", "delta_vram_gib": 0.58, "quality_lift": 0.07, "cold_load_ms": 210},
    {"adapter": "safety-lora", "route": "llama-structured-sglang", "domain": "safety", "delta_vram_gib": 0.61, "quality_lift": 0.05, "cold_load_ms": 180},
])

print(base_models[["route", "runtime", "base_vram_gib", "base_latency_ms", "throughput_tps"]].to_markdown(index=False))
print(adapters.to_markdown(index=False))

## Step 3: Define traffic classes and tenant mixes

The routing layer should respond to workload shape, not just model popularity. Different tenants and request classes generate very different adapter pressure.

In [ ]:
traffic_classes = pd.DataFrame([
    {"request_class": "general_chat", "domain": "general", "prompt_tokens": 850, "output_tokens": 180, "strict_schema": False, "long_context": False, "latency_slo_ms": 1400, "local_only": False},
    {"request_class": "support_triage", "domain": "support", "prompt_tokens": 1200, "output_tokens": 150, "strict_schema": True, "long_context": False, "latency_slo_ms": 1500, "local_only": False},
    {"request_class": "finance_extract", "domain": "finance", "prompt_tokens": 1800, "output_tokens": 120, "strict_schema": True, "long_context": False, "latency_slo_ms": 1700, "local_only": False},
    {"request_class": "sql_agent", "domain": "sql", "prompt_tokens": 1500, "output_tokens": 220, "strict_schema": True, "long_context": False, "latency_slo_ms": 1600, "local_only": False},
    {"request_class": "safety_review", "domain": "safety", "prompt_tokens": 2200, "output_tokens": 140, "strict_schema": True, "long_context": True, "latency_slo_ms": 1850, "local_only": False},
    {"request_class": "local_draft", "domain": "general", "prompt_tokens": 700, "output_tokens": 90, "strict_schema": False, "long_context": False, "latency_slo_ms": 2200, "local_only": True},
])
traffic_classes["total_tokens"] = traffic_classes["prompt_tokens"] + traffic_classes["output_tokens"]

tenant_profiles = pd.DataFrame([
    {"tenant": "castalia-mentor", "traffic_share": 0.52, "mix": {"general_chat": 0.52, "support_triage": 0.24, "finance_extract": 0.08, "sql_agent": 0.08, "safety_review": 0.03, "local_draft": 0.05}},
    {"tenant": "analyst-console", "traffic_share": 0.28, "mix": {"general_chat": 0.18, "support_triage": 0.08, "finance_extract": 0.34, "sql_agent": 0.22, "safety_review": 0.10, "local_draft": 0.08}},
    {"tenant": "ops-guardrail", "traffic_share": 0.20, "mix": {"general_chat": 0.16, "support_triage": 0.12, "finance_extract": 0.04, "sql_agent": 0.14, "safety_review": 0.42, "local_draft": 0.12}},
])

print(traffic_classes.to_markdown(index=False))
print(tenant_profiles[["tenant", "traffic_share"]].to_markdown(index=False))

## Step 4: Compare serving-policy footprints

Multi-LoRA is attractive because it reduces adapter churn, but it still consumes memory. We will compare memory and residency assumptions for four policies.

In [ ]:
policy_configs = {
    "base_only": {"loaded_limit": 0, "cold_multiplier": 0.0, "warm_adapter_ms": 0.0, "quality_bonus": 0.0, "queue_factor": 1.08, "manager_vram_gib": 0.0},
    "hot_swap": {"loaded_limit": 1, "cold_multiplier": 1.0, "warm_adapter_ms": 16.0, "quality_bonus": 0.01, "queue_factor": 1.02, "manager_vram_gib": 0.8},
    "multi_lora": {"loaded_limit": 4, "cold_multiplier": 0.45, "warm_adapter_ms": 10.0, "quality_bonus": 0.012, "queue_factor": 0.94, "manager_vram_gib": 1.6},
    "tenant_partitioned": {"loaded_limit": 2, "cold_multiplier": 0.18, "warm_adapter_ms": 8.0, "quality_bonus": 0.02, "queue_factor": 0.98, "manager_vram_gib": 4.2},
}

def estimate_policy_footprint(policy_name):
    cfg = policy_configs[policy_name]
    shared_base_vram = base_models[base_models["route"] != "mistral-edge-llamacpp"]["base_vram_gib"].sum()
    edge_vram = float(base_models.loc[base_models["route"] == "mistral-edge-llamacpp", "base_vram_gib"].iloc[0])
    if policy_name == "base_only":
        adapter_vram = 0.0
    elif policy_name == "hot_swap":
        adapter_vram = float(adapters["delta_vram_gib"].max())
    elif policy_name == "multi_lora":
        adapter_vram = float(adapters["delta_vram_gib"].sum())
    else:
        adapter_vram = float(adapters["delta_vram_gib"].sum() + 2.8)
    total_vram = shared_base_vram + edge_vram + adapter_vram + cfg["manager_vram_gib"]
    return {
        "policy": policy_name,
        "estimated_vram_gib": round(total_vram, 2),
        "active_adapter_slots": cfg["loaded_limit"],
        "cold_load_factor": cfg["cold_multiplier"],
        "notes": {
            "base_only": "No adapter residency; simplest but wastes finetuning gains.",
            "hot_swap": "One adapter slot per route; lowest memory but highest churn.",
            "multi_lora": "Keep the hot adapter set resident; best mixed-domain utilization.",
            "tenant_partitioned": "Reserve adapter lanes per tenant or risk class; strongest isolation but heavier memory.",
        }[policy_name],
    }

policy_footprint = pd.DataFrame([estimate_policy_footprint(name) for name in policy_configs])
policy_footprint

## Step 5: Build a routing heuristic

The heuristic below chooses a base route first, then attaches a domain adapter when one exists. Strict schemas and long-context work lean toward `SGLang`; code-heavy requests lean toward the coder route; local drafts stay on `llama.cpp`.

In [ ]:
route_records = {row["route"]: row for row in base_models.to_dict("records")}
request_records = {row["request_class"]: row for row in traffic_classes.to_dict("records")}
adapter_lookup = {(row["route"], row["domain"]): row for row in adapters.to_dict("records")}

def quality_column(request):
    if request["domain"] == "sql":
        return "quality_code"
    if request["strict_schema"]:
        return "quality_structured"
    return "quality_general"

def choose_base_route(request):
    best = None
    best_score = -1e9
    for row in base_models.to_dict("records"):
        if request["local_only"] and not row["local_capable"]:
            continue
        score = row[quality_column(request)]
        if request["domain"] in {"support", "finance"} and row["route"] == "qwen-general-vllm":
            score += 0.03
        if request["domain"] == "sql" and row["route"] == "qwen-coder-vllm":
            score += 0.07
        if request["strict_schema"] and row["runtime"] == "SGLang":
            score += 0.04
        if request["long_context"] and row["context_tokens"] >= 65536:
            score += 0.03
        if request["local_only"] and row["runtime"] == "llama.cpp":
            score += 0.06
        if not request["local_only"] and row["runtime"] == "llama.cpp":
            score -= 0.12
        score += min(0.05, row["throughput_tps"] / 4000.0)
        if request["latency_slo_ms"] < row["base_latency_ms"] + 250:
            score -= 0.02
        if score > best_score:
            best = row
            best_score = score
    return best

def target_adapter_for_request(route, request):
    return adapter_lookup.get((route["route"], request["domain"]))

routing_examples = []
for request in traffic_classes.to_dict("records"):
    route = choose_base_route(request)
    adapter = target_adapter_for_request(route, request)
    routing_examples.append({
        "request_class": request["request_class"],
        "domain": request["domain"],
        "recommended_route": route["route"],
        "runtime": route["runtime"],
        "adapter": "" if adapter is None else adapter["adapter"],
    })

pd.DataFrame(routing_examples)

## Step 6: Simulate several serving policies under load

We will compare four policies:

- `base_only`: no adapter residency
- `hot_swap`: one adapter slot per route
- `multi_lora`: keep the hot set resident on a shared runtime
- `tenant_partitioned`: isolate tenant or risk-heavy traffic with smaller per-lane adapter sets

In [ ]:
tenant_weight_map = {row["tenant"]: row["traffic_share"] for row in tenant_profiles.to_dict("records")}
tenant_mix_map = {row["tenant"]: row["mix"] for row in tenant_profiles.to_dict("records")}

def sample_request(rng):
    tenant = weighted_pick(tenant_weight_map, rng)
    request_class = weighted_pick(tenant_mix_map[tenant], rng)
    record = copy.deepcopy(request_records[request_class])
    record["tenant"] = tenant
    return record

def simulate_policy(policy_name, n_requests=360, seed=0):
    cfg = policy_configs[policy_name]
    rng = random.Random(seed + len(policy_name))
    adapter_cache = defaultdict(deque)
    arrival_clock = 0.0
    server_free_ms = 0.0
    rows = []
    for request_id in range(n_requests):
        arrival_clock += rng.expovariate(2.9) * 1000.0
        request = sample_request(rng)
        route = choose_base_route(request)
        adapter = target_adapter_for_request(route, request) if cfg["loaded_limit"] > 0 else None
        adapter_name = None
        adapter_hit = False
        cold_load_ms = 0.0
        if adapter is not None:
            adapter_name = adapter["adapter"]
            cache_key = route["route"] if policy_name != "tenant_partitioned" else route["route"] + "::" + request["tenant"]
            cache = adapter_cache[cache_key]
            if adapter_name in cache:
                adapter_hit = True
                cache.remove(adapter_name)
                cache.append(adapter_name)
            else:
                cold_load_ms = adapter["cold_load_ms"] * cfg["cold_multiplier"]
                if len(cache) >= cfg["loaded_limit"] and len(cache) > 0:
                    cache.popleft()
                cache.append(adapter_name)
        token_scale = 0.62 + request["total_tokens"] / 1700.0
        base_latency_ms = route["base_latency_ms"] * token_scale
        if request["strict_schema"] and route["runtime"] == "SGLang":
            base_latency_ms *= 0.95
        if request["long_context"] and route["context_tokens"] >= 65536:
            base_latency_ms *= 0.94
        if request["local_only"] and route["runtime"] == "llama.cpp":
            base_latency_ms *= 0.90
        latency_ms = max(140.0, base_latency_ms + cold_load_ms + (cfg["warm_adapter_ms"] if adapter_name else 0.0) + rng.uniform(-35, 35))
        service_ms = latency_ms * cfg["queue_factor"] * (0.72 if route["runtime"] in ["vLLM", "SGLang"] else 0.88)
        start_ms = max(arrival_clock, server_free_ms)
        wait_ms = start_ms - arrival_clock
        finish_ms = start_ms + service_ms
        server_free_ms = finish_ms
        quality = route[quality_column(request)]
        if adapter is not None:
            quality += adapter["quality_lift"] + cfg["quality_bonus"]
        if request["strict_schema"] and route["runtime"] == "SGLang":
            quality += 0.02
        if request["domain"] == "sql" and route["route"] == "qwen-coder-vllm":
            quality += 0.02
        quality = clipped(quality, 0.0, 0.99)
        total_latency_ms = wait_ms + latency_ms
        slo_met = total_latency_ms <= request["latency_slo_ms"]
        pass_bias = 0.78 + (quality - 0.80) * 1.8 + (0.04 if adapter_name else 0.0)
        if request["strict_schema"] and route["runtime"] == "SGLang":
            pass_bias += 0.03
        if not slo_met:
            pass_bias -= 0.10
        contract_pass = rng.random() < clipped(pass_bias, 0.05, 0.995)
        rows.append({
            "policy": policy_name,
            "request_id": request_id,
            "tenant": request["tenant"],
            "request_class": request["request_class"],
            "domain": request["domain"],
            "route": route["route"],
            "runtime": route["runtime"],
            "used_adapter": adapter_name,
            "adapter_hit": adapter_hit,
            "cold_load_ms": round(cold_load_ms, 2),
            "latency_ms": round(latency_ms, 2),
            "wait_ms": round(wait_ms, 2),
            "total_latency_ms": round(total_latency_ms, 2),
            "quality_score": round(quality, 4),
            "contract_pass": contract_pass,
            "slo_met": slo_met,
        })
    return pd.DataFrame(rows)

simulation_df = pd.concat([simulate_policy(policy_name, seed=13) for policy_name in policy_configs], ignore_index=True)
simulation_df.head(10)

## Step 7: Summarize policy tradeoffs

The important metrics are contract pass rate, latency, SLO pass rate, adapter hit rate, and cold-load frequency. No single policy wins every workload.

In [ ]:
policy_summary = (
    simulation_df.groupby("policy")
    .agg(
        requests=("request_id", "count"),
        mean_latency_ms=("total_latency_ms", "mean"),
        p95_latency_ms=("total_latency_ms", lambda values: percentile(values, 95)),
        mean_quality=("quality_score", "mean"),
        contract_pass_rate=("contract_pass", "mean"),
        slo_pass_rate=("slo_met", "mean"),
        adapter_use_rate=("used_adapter", lambda values: float(pd.Series(values).notna().mean())),
        adapter_hit_rate=("adapter_hit", "mean"),
        cold_load_rate=("cold_load_ms", lambda values: float((pd.Series(values) > 0).mean())),
    )
    .round(3)
    .reset_index()
)

throughput_rows = []
for policy_name, group in simulation_df.groupby("policy"):
    horizon_ms = float(group["total_latency_ms"].sum())
    throughput_rows.append({"policy": policy_name, "throughput_rps": round(len(group) / max(horizon_ms / 1000.0, 0.001), 3)})

policy_summary = policy_summary.merge(pd.DataFrame(throughput_rows), on="policy", how="left")
policy_summary.sort_values(["contract_pass_rate", "p95_latency_ms"], ascending=[False, True])

## Step 8: Visualize the latency and quality tradeoff

Packed multi-LoRA often improves adapter residency without paying the full isolation tax. Tenant partitioning may still win when strict schemas or safety traffic justify the extra overhead.

In [ ]:
chart = policy_summary.sort_values("p95_latency_ms")
fig, ax = plt.subplots(figsize=(7.8, 5.0))
scatter = ax.scatter(chart["p95_latency_ms"], chart["contract_pass_rate"], s=200 + 600 * chart["adapter_hit_rate"], c=chart["mean_quality"], cmap="viridis")
for _, row in chart.iterrows():
    ax.text(row["p95_latency_ms"] + 6, row["contract_pass_rate"] + 0.002, row["policy"])
ax.set_xlabel("p95 latency ms")
ax.set_ylabel("contract pass rate")
ax.set_title("Serving policy trade-offs: latency, quality, and adapter residency")
plt.colorbar(scatter, ax=ax, label="mean quality")
plt.show()

## Step 9: Inspect traffic partitioning by tenant and route

Traffic partitioning is not only about performance. It is also about making heavy tenants, sensitive flows, or schema-heavy traffic legible to operators.

In [ ]:
partition_summary = (
    simulation_df.groupby(["policy", "tenant", "route"])
    .size()
    .rename("requests")
    .reset_index()
)
partition_summary["route_share"] = partition_summary.groupby(["policy", "tenant"])["requests"].transform(lambda values: values / values.sum())
partition_summary.sort_values(["policy", "tenant", "route_share"], ascending=[True, True, False]).head(24)

## Step 10: Size the adapter hot set

Multi-LoRA is valuable when the hot adapter set is smaller than the total adapter universe. An LRU sketch gives a fast intuition for how many slots you actually need.

In [ ]:
def simulate_lru_hit_rate(sequence, budget):
    cache = deque()
    hits = 0
    misses = 0
    for name in sequence:
        if name in cache:
            hits += 1
            cache.remove(name)
            cache.append(name)
        else:
            misses += 1
            if len(cache) >= budget and len(cache) > 0:
                cache.popleft()
            cache.append(name)
    total = hits + misses
    return {
        "adapter_slots": budget,
        "hit_rate": round(hits / total, 3) if total else 0.0,
        "miss_rate": round(misses / total, 3) if total else 0.0,
    }

rng = random.Random(1300)
demand_sequence = []
for _ in range(320):
    request = sample_request(rng)
    route = choose_base_route(request)
    adapter = target_adapter_for_request(route, request)
    if adapter is not None:
        demand_sequence.append(adapter["adapter"])

hotset_summary = pd.DataFrame([simulate_lru_hit_rate(demand_sequence, budget) for budget in [1, 2, 3, 4]])
hotset_summary

## Step 11: Recommend a runtime and policy from scenario shape

You do not choose `vLLM`, `SGLang`, or `llama.cpp` in isolation. You choose a combined serving stack: runtime plus adapter policy plus routing assumptions.

In [ ]:
def recommend_stack(scenario):
    if scenario["local_only"]:
        return "llama.cpp + base_only"
    if scenario["strict_schema"] and scenario["safety_critical"]:
        return "SGLang + tenant_partitioned"
    if scenario["active_domains"] >= 3 and scenario["adapter_churn"] >= 0.45:
        return "vLLM + multi_lora"
    if scenario["active_domains"] == 1 and scenario["adapter_churn"] < 0.20:
        return "vLLM + hot_swap"
    return "vLLM + multi_lora"

scenarios = pd.DataFrame([
    {"scenario": "offline laptop drafting", "local_only": True, "strict_schema": False, "safety_critical": False, "active_domains": 1, "adapter_churn": 0.05},
    {"scenario": "mixed support and finance api", "local_only": False, "strict_schema": True, "safety_critical": False, "active_domains": 2, "adapter_churn": 0.32},
    {"scenario": "multi-tenant tool platform", "local_only": False, "strict_schema": True, "safety_critical": False, "active_domains": 4, "adapter_churn": 0.58},
    {"scenario": "guardrail-heavy review service", "local_only": False, "strict_schema": True, "safety_critical": True, "active_domains": 2, "adapter_churn": 0.27},
])
scenarios["recommended_stack"] = scenarios.apply(lambda row: recommend_stack(row.to_dict()), axis=1)
scenarios

## Step 12: Export a serving manifest

The export bundles model registry, adapter assumptions, policy summaries, and scenario recommendations into a lightweight planning artifact.

In [ ]:
manifest = {
    "base_models": base_models.to_dict("records"),
    "adapters": adapters.to_dict("records"),
    "policy_footprint": policy_footprint.to_dict("records"),
    "policy_summary": policy_summary.to_dict("records"),
    "partition_summary": partition_summary.to_dict("records"),
    "hotset_summary": hotset_summary.to_dict("records"),
    "scenario_recommendations": scenarios.to_dict("records"),
}
write_json(ARTIFACT_DIR / "multi_lora_serving_manifest.json", manifest)
policy_summary.to_csv(ARTIFACT_DIR / "policy_summary.csv", index=False)
partition_summary.to_csv(ARTIFACT_DIR / "partition_summary.csv", index=False)
print("Saved serving artifacts to", ARTIFACT_DIR.resolve())

## Exercises

1. Increase the `safety_review` traffic share and see when tenant partitioning becomes clearly worth the memory cost.
2. Add a second SQL adapter for a different base model and compare whether routing simplicity or adapter lift matters more.
3. Convert the scenario recommender into a benchmark-gated deployment policy with explicit SLO thresholds.

In [ ]:
student_checks = pd.DataFrame([
    {"question": "When would hot-swap beat packed multi-LoRA?", "your_answer": ""},
    {"question": "Which tenant should receive the most isolated serving lane?", "your_answer": ""},
    {"question": "How many adapter slots are enough for this demand sequence?", "your_answer": ""},
])
student_checks

## Recap

Multi-LoRA serving is a routing problem as much as a modeling problem. The right answer depends on adapter churn, tenant mix, schema strictness, and whether you value utilization or isolation more. Runtime choice follows the same rule: use the stack whose scheduling model matches the traffic you actually have.

In [ ]:
best_policy = policy_summary.sort_values(["contract_pass_rate", "p95_latency_ms"], ascending=[False, True]).iloc[0]["policy"]
assert best_policy in {"multi_lora", "tenant_partitioned"}
assert hotset_summary["hit_rate"].iloc[-1] >= hotset_summary["hit_rate"].iloc[0]
assert "llama.cpp + base_only" in scenarios["recommended_stack"].tolist()
assert simulation_df["used_adapter"].notna().any()
print("Notebook 13 sanity checks passed.")